# Strain sobre `relax_final.xyz`
Genera `cases/<caso>/` (estructura `case.xyz` + `input.in` de QE) con deformación **uniaxial en x**, **uniaxial en y** y **biaxial**, aleatoria en [`EMIN`, `EMAX`] %. Después copia la carpeta al cluster y usa `02_submit_scf.sh`.

Al final, la sección **Biaxial anisótropo** añade casos con εxx y εyy independientes, sin tocar los anteriores.

In [ ]:
BASE = "relax_final.xyz"
N = 30                  # valores aleatorios por modo (3 modos -> 90 casos)
EMIN, EMAX = -25, 25    # rango de deformación (%)
SEED = 1996             # semilla: pon el número que quieras (mismo número = mismos valores)
PBC = (True, True, False)

In [ ]:
# Input de QE (los ajustes de tu input_nvt.in, pasados a scf). {nat}, {cell}... se rellenan por caso.
SPECIES = {"C": (12.011, "C.pbe-n-kjpaw_psl.1.0.0.UPF"), "B": (10.811, "B.pbe-n-kjpaw_psl.1.0.0.UPF")}

INPUT = """&CONTROL
 calculation = 'scf'
 pseudo_dir = '../../pseudos'
 outdir = './save'
 tprnfor = .true.
 tstress = .true.
 disk_io = 'none'   ! sin wfc ni densidad en disco (solo hacen falta E, F y tension); si pw.x lo rechaza: 'nowf'
/
&SYSTEM
 ibrav = 0
 nat = {nat}
 ntyp = {ntyp}
 nspin = 2
 starting_magnetization(1) = 0.1d0
 starting_magnetization(2) = 0.1d0
 ecutwfc = 60.0
 ecutrho = 600.0
 vdw_corr = 'Grimme-D3'
 occupations = 'smearing'
 smearing = 'gaussian'
 degauss = 0.005
 nosym = .true.
/
&ELECTRONS
 electron_maxstep = 250
 mixing_mode = 'plain'
 mixing_beta = 0.2
 conv_thr = 7.35d-6
/
ATOMIC_SPECIES
{species}
CELL_PARAMETERS angstrom
{cell}
ATOMIC_POSITIONS angstrom
{pos}
K_POINTS gamma
"""

In [ ]:
import shutil
from pathlib import Path

import numpy as np
from ase import Atoms
from ase.io import read, write

b = read(BASE)
base = Atoms(b.symbols, b.positions, cell=b.cell, pbc=PBC)     # limpio: sin la energía/step del relax
assert set(base.symbols) <= set(SPECIES), "añade la especie a SPECIES"

rng = np.random.default_rng(SEED)
edges = np.linspace(EMIN, EMAX, N + 1)
W = {"uni-x": (1, 0), "uni-y": (0, 1), "biaxial": (1, 1)}      # pesos de (e_xx, e_yy)
# un valor aleatorio dentro de cada intervalo de [EMIN, EMAX]: aleatorio, pero sin huecos
casos = [(m, float(e)) for m in W for e in np.round(rng.uniform(edges[:-1], edges[1:]), 4)]

out = Path("cases")
assert not list(out.glob("*/qe_out.out")), "cases/ ya tiene resultados: muévela antes de regenerar"
shutil.rmtree(out, ignore_errors=True)
for k, (m, e) in enumerate(casos):
    exx, eyy = W[m][0] * e + 0.0, W[m][1] * e + 0.0                              # + 0.0 evita -0.0
    name = f"{m}_{'m' if e < 0 else 'p'}{abs(e):05.2f}"
    a = base.copy()
    a.set_cell(a.cell.array @ np.diag([1 + exx / 100, 1 + eyy / 100, 1]), scale_atoms=True)
    a.info.update(case=name, strain_mode=m, strain_xx_pct=exx, strain_yy_pct=eyy, seed=SEED)
    d = out / f"case_{k:02d}_{name}"
    d.mkdir(parents=True)
    write(d / "case.xyz", a)
    txt = INPUT.format(nat=len(a), ntyp=len(SPECIES),
                       species="\n".join(f"{s} {mass} {upf}" for s, (mass, upf) in SPECIES.items()),
                       cell="\n".join("%.10f %.10f %.10f" % tuple(v) for v in a.cell.array),
                       pos="\n".join("%-2s %.10f %.10f %.10f" % (s, *p) for s, p in zip(a.symbols, a.positions)))
    with open(d / "input.in", "w", newline="\n") as f:
        f.write(txt)

print(f"semilla {SEED} · {len(casos)} casos en cases/")
for m in W:
    print(f"{m:8s}", " ".join(f"{e:+.2f}" for mm, e in casos if mm == m))

## Biaxial anisótropo (εxx ≠ εyy)
Añade a `cases/` casos con εxx y εyy **independientes**. No borra nada ni toca los 90 anteriores, y se para si una carpeta nueva fuera a pisar otra distinta o con resultados. Basta con ejecutar las dos primeras celdas (parámetros e input de QE) y las de esta sección: no hace falta repetir la de los 90 casos.

- **Muestreo**: hipercubo latino, la versión 2D de lo de arriba. En cada eje cae un valor aleatorio dentro de cada uno de los `N_XY` intervalos, y se emparejan al azar. De muchos sorteos se queda con el de puntos más repartidos (mayor distancia mínima), para que no queden grumos ni huecos en el plano.
- **Modo** `aniso`: sin «biaxial» ni «uni» en el nombre, para que no se mezcle con los anteriores si algún script filtra por nombre.
- **Opcional**: cizalla εxy (`EXY_MAX`) y desplazamientos aleatorios en x, y, z (`RATTLE`) sobre cada celda deformada. Con `0` quedan apagados.

In [ ]:
BASE = "relax_final.xyz"
PBC = (True, True, False)

N_XY = 60                   # casos nuevos
EMIN_XY, EMAX_XY = -20, 20  # rango de εxx y de εyy (%), cada uno por su lado
EXY_MAX = 0                 # cizalla: εxy en [-EXY_MAX, EXY_MAX] %        (0 = sin cizalla)
RATTLE = 0.0                # σ (Å) de desplazamientos aleatorios en x, y, z (0 = sin rattling; p. ej. 0.03)
SEED_XY = 2026              # semilla propia: no depende de la de arriba
K0 = 90                     # primer índice de carpeta (case_00 … case_89 ya existen)
MODO_XY = "aniso"

In [ ]:
# Input de QE (los ajustes de tu input_nvt.in, pasados a scf). {nat}, {cell}... se rellenan por caso.
SPECIES = {"C": (12.011, "C.pbe-n-kjpaw_psl.1.0.0.UPF"), "B": (10.811, "B.pbe-n-kjpaw_psl.1.0.0.UPF")}

INPUT = """&CONTROL
 calculation = 'scf'
 pseudo_dir = '../../pseudos'
 outdir = './save'
 tprnfor = .true.
 tstress = .true.
 disk_io = 'none'   ! sin wfc ni densidad en disco (solo hacen falta E, F y tension); si pw.x lo rechaza: 'nowf'
/
&SYSTEM
 ibrav = 0
 nat = {nat}
 ntyp = {ntyp}
 nspin = 2
 starting_magnetization(1) = 0.1d0
 starting_magnetization(2) = 0.1d0
 ecutwfc = 60.0
 ecutrho = 600.0
 vdw_corr = 'Grimme-D3'
 occupations = 'smearing'
 smearing = 'gaussian'
 degauss = 0.005
 nosym = .true.
/
&ELECTRONS
 electron_maxstep = 250
 mixing_mode = 'plain'
 mixing_beta = 0.2
 conv_thr = 7.35d-6
/
ATOMIC_SPECIES
{species}
CELL_PARAMETERS angstrom
{cell}
ATOMIC_POSITIONS angstrom
{pos}
K_POINTS gamma
"""

In [ ]:
from pathlib import Path

import numpy as np
from ase import Atoms
from ase.io import read, write

b = read(BASE)
base = Atoms(b.symbols, b.positions, cell=b.cell, pbc=PBC)
assert set(base.symbols) <= set(SPECIES), "añade la especie a SPECIES"

def lhs(n, d, rng, intentos=2000):
    """n puntos en [0, 1)^d: en cada eje, uno al azar dentro de cada uno de los n intervalos (hipercubo latino).
    De `intentos` sorteos se queda con el de mayor distancia mínima entre puntos."""
    iu = np.triu_indices(n, 1)
    mejor, dmin = None, -1.0
    for _ in range(intentos):
        u = (np.argsort(rng.random((d, n)), axis=1).T + rng.random((n, d))) / n
        dist = np.sqrt(((u[:, None] - u[None]) ** 2).sum(-1))[iu].min()
        if dist > dmin:
            mejor, dmin = u, dist
    return mejor

rng = np.random.default_rng(SEED_XY)
lo = np.array([EMIN_XY, EMIN_XY] + ([-EXY_MAX] if EXY_MAX else []), float)
hi = np.array([EMAX_XY, EMAX_XY] + ([EXY_MAX] if EXY_MAX else []), float)
eps = np.round(lo + (hi - lo) * lhs(N_XY, len(lo), rng), 4) + 0.0          # + 0.0 evita -0.0
if not EXY_MAX:
    eps = np.c_[eps, np.zeros(N_XY)]                                       # columnas: εxx, εyy, εxy (%)

sg = lambda e: f"{'m' if e < 0 else 'p'}{abs(e):05.2f}"
nombres = [f"case_{k:02d}_{MODO_XY}_x{sg(x)}_y{sg(y)}" + (f"_s{sg(s)}" if EXY_MAX else "")
           for k, (x, y, s) in enumerate(eps, start=K0)]

out = Path("cases")
for k, nom in enumerate(nombres, start=K0):           # seguridad: solo se rehace esta misma tanda, y sin resultados
    for p in out.glob(f"case_{k:02d}_*"):
        assert p.name == nom, f"{p} ya existe con otro nombre: cambia K0 o bórrala a mano"
        assert not (p / "qe_out.out").exists(), f"{p} ya tiene resultados: no la regenero"

for nom, (exx, eyy, exy) in zip(nombres, eps):
    a = base.copy()
    D = np.eye(3) + np.array([[exx, exy, 0], [exy, eyy, 0], [0, 0, 0]]) / 100     # deformación simétrica (z intacto)
    a.set_cell(a.cell.array @ D, scale_atoms=True)
    if RATTLE:
        a.positions += rng.normal(0, RATTLE, a.positions.shape)
    a.info.update(case=nom.split("_", 2)[2], strain_mode=MODO_XY, strain_xx_pct=exx, strain_yy_pct=eyy,
                  strain_xy_pct=exy, rattle_A=RATTLE, seed=SEED_XY)
    d = out / nom
    d.mkdir(parents=True, exist_ok=True)
    write(d / "case.xyz", a)
    txt = INPUT.format(nat=len(a), ntyp=len(SPECIES),
                       species="\n".join(f"{s} {mass} {upf}" for s, (mass, upf) in SPECIES.items()),
                       cell="\n".join("%.10f %.10f %.10f" % tuple(v) for v in a.cell.array),
                       pos="\n".join("%-2s %.10f %.10f %.10f" % (s, *p) for s, p in zip(a.symbols, a.positions)))
    with open(d / "input.in", "w", newline="\n") as f:
        f.write(txt)

x, y = eps[:, 0], eps[:, 1]
area = ((1 + x / 100) * (1 + y / 100) - (eps[:, 2] / 100) ** 2 - 1) * 100
print(f"semilla {SEED_XY} · {N_XY} casos nuevos en {out}/: {nombres[0]} … {nombres[-1]}")
print(f"cuadrantes: (+,+) {((x > 0) & (y > 0)).sum()} · (−,−) {((x < 0) & (y < 0)).sum()} · "
      f"(+,−) {((x > 0) & (y < 0)).sum()} · (−,+) {((x < 0) & (y > 0)).sum()}")
print(f"cambio de área: de {area.min():+.1f} % a {area.max():+.1f} %" + (f" · rattling σ = {RATTLE} Å" if RATTLE else ""))
for nom, a_ in zip(nombres, area):
    print(f"  {nom:46s} área {a_:+6.1f} %")

### Comprobación: el plano (εxx, εyy) con todos los casos de `cases/`
Lee los `case.xyz` del disco, así que muestra lo que realmente vas a mandar al cluster. Los 90 anteriores son tres rectas (los dos ejes y la diagonal); los nuevos deberían rellenar el cuadro discontinuo (`EMIN_XY` … `EMAX_XY`) sin grumos.

In [ ]:
import matplotlib.pyplot as plt

SUP, TINTA2, TENUE, REJ, NUEVO = "#fcfcfb", "#52514e", "#898781", "#e1e0d9", "#2a78d6"
casos_disco = [read(p).info for p in sorted(Path("cases").glob("case_*/case.xyz"))]
es_nuevo = np.array([c["strain_mode"] == MODO_XY for c in casos_disco])
E2 = np.array([(c["strain_xx_pct"], c["strain_yy_pct"]) for c in casos_disco])

fig, ax = plt.subplots(figsize=(5.6, 5.6), facecolor=SUP)
ax.set_facecolor(SUP)
ax.add_patch(plt.Rectangle((EMIN_XY, EMIN_XY), EMAX_XY - EMIN_XY, EMAX_XY - EMIN_XY, fill=False, ls="--", lw=1, ec=TENUE, zorder=1))
ax.scatter(*E2[~es_nuevo].T, s=34, marker="o", color="#c3c2b7", edgecolor=SUP, lw=0.8, zorder=2,
           label=f"ya calculados ({(~es_nuevo).sum()})")
ax.scatter(*E2[es_nuevo].T, s=58, marker="D", color=NUEVO, edgecolor=SUP, lw=0.8, zorder=3,
           label=f"nuevos · {MODO_XY} ({es_nuevo.sum()})")
lim = 1.12 * np.abs(E2).max()
ax.set(xlim=(-lim, lim), ylim=(-lim, lim), xlabel="εxx (%)", ylabel="εyy (%)", aspect="equal")
ax.set_title("Plano de deformaciones", loc="left", fontsize=12, fontweight="bold", pad=26)
ax.grid(color=REJ, lw=0.8)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color("#c3c2b7")
ax.tick_params(colors=TENUE)
ax.xaxis.label.set_color(TINTA2)
ax.yaxis.label.set_color(TINTA2)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, fontsize=9, labelcolor=TINTA2,
          borderaxespad=0.2, handletextpad=0.3, columnspacing=1.2)
fig.tight_layout()
fig.savefig("plano_deformaciones.png", dpi=150)
plt.show()